<a href="https://colab.research.google.com/github/vrajkmrpatel/NLP-GENAI-TASKS/blob/main/Task_B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers groq scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.9 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from groq import Groq
import os

In [ ]:
os.environ["GROQ_API_KEY"] = ""
client = Groq()

In [ ]:
chunks = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model.",
    "The retrieval step finds relevant documents using vector similarity search.",
    "Embeddings convert text into dense numerical vectors that capture semantic meaning.",
    "LLMs can hallucinate facts that are not present in their training data.",
    "Fine-tuning updates model weights, while RAG keeps the model frozen and retrieves context at inference time."
]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chunk_embeddings = model.encode(chunks)

In [ ]:
query = "What is the difference between RAG and fine-tuning?"
query_embedding = model.encode([query])

In [ ]:
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

In [ ]:
top_k = 2
top_indices = np.argsort(similarities)[-top_k:][::-1]

retrieved_chunks = [chunks[i] for i in top_indices]

print("Retrieved Chunks:\n")
for chunk in retrieved_chunks:
    print("-", chunk)

Retrieved Chunks:

- Fine-tuning updates model weights, while RAG keeps the model frozen and retrieves context at inference time.
- RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model.


In [ ]:
context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.5
)

answer = response.choices[0].message.content

In [ ]:
print("\nFinal Answer:\n")
print(answer)


Final Answer:

The main difference between RAG (Retrieval-Augmented Generation) and fine-tuning is how they update and utilize the model. Fine-tuning updates the model weights, whereas RAG keeps the model frozen (i.e., the model weights remain unchanged) and instead retrieves context at inference time to generate outputs.
